In [ ]:
-- create a database
 create or replace database anycompany_lab;

In [ ]:
 -- create a schema bronze
create or replace schema anycompany_lab.bronze;


In [ ]:
 -- create a schema silver
create or replace schema anycompany_lab.silver;

In [ ]:
-- create stage
 create or replace stage anycompany_lab.bronze.food_beverage_stage
  URL = 's3://logbrain-datalake/datasets/food-beverage/';

   list @anycompany_lab.bronze.food_beverage_stage

## Étape 2 – Création des tables

### Dans le schéma bronze

Tables csv

In [ ]:
create or replace table anycompany_lab.bronze.customer_demographics (
customer_id string,
name string,
date_of_birth string,
gender string,
region string,
country string,
city string,
marital_status string,
annual_income string
);

create or replace table anycompany_lab.bronze.customer_service_interactions(
interaction_id  string,
interaction_date string,
interaction_type string,
issue_category string,
description string,
duration_minutes string,
resolution_status string,
follow_up_required string,
customer_satisfaction string
);

create or replace table anycompany_lab.bronze.financial_transactions(
transaction_id string,
transaction_date string,
transaction_type string,
amount string,
payment_method string,
entity string,
region string,
account_code string
);

create or replace table anycompany_lab.bronze.promotions_data(
promotion_id string,
product_category string,
promotion_type string,
discount_percentage string,
start_date string,
end_date string,
region string
);

create or replace table anycompany_lab.bronze.marketing_campaigns(
campaign_id  string,
campaign_name string,
campaign_type string,
product_category string,
target_audience string,
start_date string,
end_date string,
region string,
budget string,
reach string,
conversion_rate string
);

create or replace table anycompany_lab.bronze.product_reviews(
review_id string,
product_id string,
reviewer_id string,
reviewer_name string,
rating string,
review_date string,
review_title string,
review_text string,
product_category string default 'UNKNOWN'
);

create or replace table anycompany_lab.bronze.logistics_and_shipping(
shipment_id string,
order_id string,
ship_date string,
estimated_delivery string,
shipping_method string,
status string,
shipping_cost string,
destination_region string,
destination_country string,
carrier string
);

create or replace table anycompany_lab.bronze.supplier_information(
supplier_id string,
supplier_name string,
product_category string,
region string,
country string,
city string,
lead_time string,
reliability_score string,
quality_rating string
);

create or replace table anycompany_lab.bronze.employee_records(
employee_id string,
name string,
date_of_birth string,
hire_date string,
department string,
job_title string,
salary string,
region string,
country string,
email string
);

In [ ]:
select * from anycompany_lab.bronze.customer_demographics 

In [ ]:
select * from anycompany_lab.bronze.customer_service_interactions

In [ ]:
select * from anycompany_lab.bronze.financial_transactions

In [ ]:
select * from anycompany_lab.bronze.promotions_data

In [ ]:
select * from anycompany_lab.bronze.marketing_campaigns

In [ ]:
select * from anycompany_lab.bronze.product_reviews

In [ ]:
select * from anycompany_lab.bronze.logistics_and_shipping

In [ ]:
select * from anycompany_lab.bronze.supplier_information

In [ ]:
select * from anycompany_lab.bronze.employee_records

Tables Json

In [ ]:
create or replace table anycompany_lab.bronze.product_reviews_(v variant)

In [ ]:
create or replace table anycompany_lab.bronze.inventory(
data variant
);

create or replace table anycompany_lab.bronze.store_locations(
data variant
);

## Étape 3 – Chargement des données


Création des formats de fichiers

In [ ]:
CREATE OR REPLACE FILE FORMAT anycompany_lab.bronze.csv
  type = 'CSV'
  field_delimiter = ','
  record_delimiter = '\n'
  skip_header = 1
  field_optionally_enclosed_by = '\042'
  null_if = ('');

create or replace file format anycompany_lab.bronze.json
  type = 'JSON'
  strip_outer_array = true;

Copy into

In [ ]:
-- copy data into csv tables:

-- customer_demographics
copy into anycompany_lab.bronze.customer_demographics
from @anycompany_lab.bronze.food_beverage_stage/customer_demographics.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;

-- customer_service_interactions
copy into anycompany_lab.bronze.customer_service_interactions
from @anycompany_lab.bronze.food_beverage_stage/customer_service_interactions.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;

-- financial_transactions
copy into anycompany_lab.bronze.financial_transactions
from @anycompany_lab.bronze.food_beverage_stage/financial_transactions.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;

-- promotions-data
copy into anycompany_lab.bronze.promotions_data
from @anycompany_lab.bronze.food_beverage_stage/promotions-data.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;

-- marketing_campaigns
copy into anycompany_lab.bronze.marketing_campaigns
from @anycompany_lab.bronze.food_beverage_stage/marketing_campaigns.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;

-- product_reviews
copy into anycompany_lab.bronze.product_reviews
from @anycompany_lab.bronze.food_beverage_stage/product_reviews.csv 
FILE_FORMAT = anycompany_lab.bronze.csv
ON_ERROR = 'CONTINUE';

-- logistics_and_shipping
copy into anycompany_lab.bronze.logistics_and_shipping
from @anycompany_lab.bronze.food_beverage_stage/logistics_and_shipping.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;

--supplier_information
copy into anycompany_lab.bronze.supplier_information
from @anycompany_lab.bronze.food_beverage_stage/supplier_information.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;

-- employee_records
copy into anycompany_lab.bronze.employee_records
from @anycompany_lab.bronze.food_beverage_stage/employee_records.csv 
FILE_FORMAT = anycompany_lab.bronze.csv;



-- copy data into json tables

--
copy into anycompany_lab.bronze.inventory
from @anycompany_lab.bronze.food_beverage_stage/inventory.json
FILE_FORMAT = anycompany_lab.bronze.json;

--
copy into anycompany_lab.bronze.store_locations
from @anycompany_lab.bronze.food_beverage_stage/store_locations.json
FILE_FORMAT = anycompany_lab.bronze.json;

In [ ]:
select * from anycompany_lab.bronze.product_reviews


In [ ]:
select * from anycompany_lab.bronze.inventory


In [ ]:
select * from anycompany_lab.bronze.store_locations

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(interaction_id) AS NULL_INTERACTION_ID,
    COUNT(*) - COUNT(interaction_date) AS NULL_DATE,
    COUNT(*) - COUNT(interaction_type) AS NULL_TYPE,
    COUNT(*) - COUNT(issue_category) AS NULL_CATEGORY,
    COUNT(*) - COUNT(description) AS NULL_DESC,
    COUNT(*) - COUNT(duration_minutes) AS NULL_DURATION,
    COUNT(*) - COUNT(resolution_status) AS NULL_STATUS,
    COUNT(*) - COUNT(follow_up_required) AS NULL_FOLLOW_UP,
    COUNT(*) - COUNT(customer_satisfaction) AS NULL_SATISFACTION
FROM anycompany_lab.bronze.customer_service_interactions;

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(transaction_id) AS NULL_TX_ID,
    COUNT(*) - COUNT(transaction_date) AS NULL_TX_DATE,
    COUNT(*) - COUNT(transaction_type) AS NULL_TX_TYPE,
    COUNT(*) - COUNT(amount) AS NULL_AMOUNT,
    COUNT(*) - COUNT(payment_method) AS NULL_PAYMENT_METHOD,
    COUNT(*) - COUNT(entity) AS NULL_ENTITY,
    COUNT(*) - COUNT(region) AS NULL_REGION,
    COUNT(*) - COUNT(account_code) AS NULL_ACCOUNT_CODE
FROM anycompany_lab.bronze.financial_transactions;

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(promotion_id) AS NULL_PROMO_ID,
    COUNT(*) - COUNT(product_category) AS NULL_CATEGORY,
    COUNT(*) - COUNT(promotion_type) AS NULL_TYPE,
    COUNT(*) - COUNT(discount_percentage) AS NULL_DISCOUNT,
    COUNT(*) - COUNT(start_date) AS NULL_START,
    COUNT(*) - COUNT(end_date) AS NULL_END,
    COUNT(*) - COUNT(region) AS NULL_REGION
FROM anycompany_lab.bronze.promotions_data;

1. Customer Demographics

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(customer_id) AS NULL_CUSTOMER_ID,
    COUNT(*) - COUNT(name) AS NULL_NAME,
    COUNT(*) - COUNT(date_of_birth) AS NULL_DOB,
    COUNT(*) - COUNT(gender) AS NULL_GENDER,
    COUNT(*) - COUNT(region) AS NULL_REGION,
    COUNT(*) - COUNT(country) AS NULL_COUNTRY,
    COUNT(*) - COUNT(city) AS NULL_CITY,
    COUNT(*) - COUNT(marital_status) AS NULL_MARITAL_STATUS,
    COUNT(*) - COUNT(annual_income) AS NULL_ANNUAL_INCOME
FROM anycompany_lab.bronze.customer_demographics;

2. Customer Service Interactions

In [ ]:
SELECT * FROM anycompany_lab.bronze.customer_service_interactions 
WHERE follow_up_required IS NULL;

3. Financial Transactions

4. Promotions Data

5. Marketing Campaigns

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(campaign_id) AS NULL_CAMP_ID,
    COUNT(*) - COUNT(campaign_name) AS NULL_NAME,
    COUNT(*) - COUNT(campaign_type) AS NULL_TYPE,
    COUNT(*) - COUNT(product_category) AS NULL_CATEGORY,
    COUNT(*) - COUNT(target_audience) AS NULL_AUDIENCE,
    COUNT(*) - COUNT(start_date) AS NULL_START,
    COUNT(*) - COUNT(end_date) AS NULL_END,
    COUNT(*) - COUNT(region) AS NULL_REGION,
    COUNT(*) - COUNT(budget) AS NULL_BUDGET,
    COUNT(*) - COUNT(reach) AS NULL_REACH,
    COUNT(*) - COUNT(conversion_rate) AS NULL_CONVERSION
FROM anycompany_lab.bronze.marketing_campaigns;

6. Product Reviews

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(review_id) AS NULL_REVIEW_ID,
    COUNT(*) - COUNT(product_id) AS NULL_PRODUCT_ID,
    COUNT(*) - COUNT(reviewer_id) AS NULL_REVIEWER_ID,
    COUNT(*) - COUNT(reviewer_name) AS NULL_NAME,
    COUNT(*) - COUNT(rating) AS NULL_RATING,
    COUNT(*) - COUNT(review_date) AS NULL_DATE,
    COUNT(*) - COUNT(review_title) AS NULL_TITLE,
    COUNT(*) - COUNT(review_text) AS NULL_TEXT,
    COUNT(*) - COUNT(product_category) AS NULL_CATEGORY
FROM anycompany_lab.bronze.product_reviews;

7. Logistics and Shipping

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(shipment_id) AS NULL_SHIP_ID,
    COUNT(*) - COUNT(order_id) AS NULL_ORDER_ID,
    COUNT(*) - COUNT(ship_date) AS NULL_SHIP_DATE,
    COUNT(*) - COUNT(estimated_delivery) AS NULL_EST_DELIVERY,
    COUNT(*) - COUNT(shipping_method) AS NULL_METHOD,
    COUNT(*) - COUNT(status) AS NULL_STATUS,
    COUNT(*) - COUNT(shipping_cost) AS NULL_COST,
    COUNT(*) - COUNT(destination_region) AS NULL_DEST_REGION,
    COUNT(*) - COUNT(destination_country) AS NULL_DEST_COUNTRY,
    COUNT(*) - COUNT(carrier) AS NULL_CARRIER
FROM anycompany_lab.bronze.logistics_and_shipping;

8. Supplier Information

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(supplier_id) AS NULL_SUPPLIER_ID,
    COUNT(*) - COUNT(supplier_name) AS NULL_NAME,
    COUNT(*) - COUNT(product_category) AS NULL_CATEGORY,
    COUNT(*) - COUNT(region) AS NULL_REGION,
    COUNT(*) - COUNT(country) AS NULL_COUNTRY,
    COUNT(*) - COUNT(city) AS NULL_CITY,
    COUNT(*) - COUNT(lead_time) AS NULL_LEAD_TIME,
    COUNT(*) - COUNT(reliability_score) AS NULL_RELIABILITY,
    COUNT(*) - COUNT(quality_rating) AS NULL_QUALITY
FROM anycompany_lab.bronze.supplier_information;

9. Employee Records

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(employee_id) AS NULL_EMP_ID,
    COUNT(*) - COUNT(name) AS NULL_NAME,
    COUNT(*) - COUNT(date_of_birth) AS NULL_DOB,
    COUNT(*) - COUNT(hire_date) AS NULL_HIRE,
    COUNT(*) - COUNT(department) AS NULL_DEPT,
    COUNT(*) - COUNT(job_title) AS NULL_TITLE,
    COUNT(*) - COUNT(salary) AS NULL_SALARY,
    COUNT(*) - COUNT(region) AS NULL_REGION,
    COUNT(*) - COUNT(country) AS NULL_COUNTRY,
    COUNT(*) - COUNT(email) AS NULL_EMAIL
FROM anycompany_lab.bronze.employee_records;

# Schéma silver

In [ ]:
create or replace schema anycompany_lab.silver;

## Tables Json

Inventory

In [ ]:
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.inventory_clean
AS
SELECT
    data:country::STRING AS country,
    data:current_stock::INT AS current_stock,
    data:last_restock_date::DATE AS last_restock_date,
    data:lead_time::INT AS lead_time,
    data:product_category::STRING AS product_category,
    data:product_id::STRING AS product_id,
    data:region::STRING AS region,
    data:reorder_point::INT AS reorder_point,
    data:warehouse::STRING AS warehouse
FROM anycompany_lab.bronze.inventory;


In [ ]:
select * from anycompany_lab.silver.inventory_clean

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(country) AS NULL_COUNTRY,
    COUNT(*) - COUNT(current_stock) AS NULL_STOCK,
    COUNT(*) - COUNT(last_restock_date) AS NULL_LAST_RESTOCK,
    COUNT(*) - COUNT(lead_time) AS NULL_LEAD_TIME,
    COUNT(*) - COUNT(product_category) AS NULL_CATEGORY,
    COUNT(*) - COUNT(product_id) AS NULL_PRODUCT_ID,
    COUNT(*) - COUNT(region) AS NULL_REGION,
    COUNT(*) - COUNT(reorder_point) AS NULL_REORDER_POINT,
    COUNT(*) - COUNT(warehouse) AS NULL_WAREHOUSE
FROM anycompany_lab.silver.inventory_clean;

Store location

In [ ]:
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.store_locations_clean
AS
SELECT
    data:address::STRING AS address,
    data:city::STRING AS city,
    data:country::STRING AS country,
    data:employee_count::INT AS employee_count,
    data:postal_code::INT AS postal_code,
    data:region::STRING AS region,
    data:square_footage::FLOAT AS square_footage,
    data:store_id::STRING AS store_id,
    data:store_name::STRING AS store_name,
    data:store_type::STRING AS store_type
FROM anycompany_lab.bronze.store_locations;

In [ ]:
select * from anycompany_lab.silver.store_locations_clean

In [ ]:
SELECT 
    COUNT(*) AS TOTAL_ROWS,
    COUNT(*) - COUNT(address) AS NULL_ADDRESS,
    COUNT(*) - COUNT(city) AS NULL_CITY,
    COUNT(*) - COUNT(country) AS NULL_COUNTRY,
    COUNT(*) - COUNT(employee_count) AS NULL_EMP_COUNT,
    COUNT(*) - COUNT(postal_code) AS NULL_POSTAL,
    COUNT(*) - COUNT(region) AS NULL_REGION,
    COUNT(*) - COUNT(square_footage) AS NULL_SQ_FT,
    COUNT(*) - COUNT(store_id) AS NULL_STORE_ID,
    COUNT(*) - COUNT(store_name) AS NULL_NAME,
    COUNT(*) - COUNT(store_type) AS NULL_TYPE
FROM anycompany_lab.silver.store_locations_clean;

## Tables csv

In [ ]:
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.customer_demographics_clean
AS
SELECT
    customer_id::INTEGER             AS customer_id,
    name::VARCHAR                    AS name,
    date_of_birth::DATE              AS date_of_birth,
    gender::VARCHAR                  AS gender,
    region::VARCHAR                  AS region,
    country::VARCHAR                 AS country,
    city::VARCHAR                    AS city,
    marital_status::VARCHAR          AS marital_status,
    annual_income::NUMBER           AS annual_income
FROM anycompany_lab.bronze.customer_demographics;

In [ ]:
select * from anycompany_lab.silver.customer_demographics_clean limit 10

In [ ]:
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.customer_service_interactions_clean
AS
SELECT
    interaction_id::VARCHAR          AS interaction_id,
    interaction_date::DATE           AS interaction_date,
    interaction_type::VARCHAR        AS interaction_type,
    issue_category::VARCHAR          AS issue_category,
    description::VARCHAR             AS description,
    duration_minutes::INTEGER        AS duration_minutes,
    resolution_status::VARCHAR       AS resolution_status,
    follow_up_required::BOOLEAN      AS follow_up_required,
    customer_satisfaction::INTEGER   AS customer_satisfaction
FROM anycompany_lab.bronze.customer_service_interactions;

In [ ]:
select * from anycompany_lab.silver.customer_service_interactions_clean limit 10

In [ ]:
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.financial_transactions_clean
AS
SELECT
    transaction_id::VARCHAR          AS transaction_id,
    transaction_date::DATE           AS transaction_date,
    transaction_type::VARCHAR        AS transaction_type,
    amount::NUMBER(10,2)             AS amount,
    payment_method::VARCHAR          AS payment_method,
    entity::VARCHAR                  AS entity,
    region::VARCHAR                  AS region,
    account_code::VARCHAR            AS account_code
FROM anycompany_lab.bronze.financial_transactions;

select * from anycompany_lab.silver.financial_transactions_clean limit 10

In [ ]:
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.promotions_clean
AS
SELECT
    promotion_id::VARCHAR            AS promotion_id,
    product_category::VARCHAR        AS product_category,
    promotion_type::VARCHAR          AS promotion_type,
    discount_percentage::NUMBER(4,2) AS discount_percentage,
    start_date::DATE                 AS start_date,
    end_date::DATE                   AS end_date,
    region::VARCHAR                  AS region
FROM anycompany_lab.bronze.promotions_data;

select * from anycompany_lab.silver.promotions_clean limit 10

In [ ]:
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.marketing_campaigns_clean
AS
SELECT
    campaign_id::VARCHAR             AS campaign_id,
    campaign_name::VARCHAR           AS campaign_name,
    campaign_type::VARCHAR           AS campaign_type,
    product_category::VARCHAR        AS product_category,
    target_audience::VARCHAR         AS target_audience,
    start_date::DATE                 AS start_date,
    end_date::DATE                   AS end_date,
    region::VARCHAR                  AS region,
    budget::NUMBER(12,2)             AS budget,
    reach::INTEGER                   AS reach,
    conversion_rate::NUMBER(6,4)     AS conversion_rate
FROM anycompany_lab.bronze.marketing_campaigns;

select * from anycompany_lab.silver.marketing_campaigns_clean limit 10

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.silver.product_reviews
AS
SELECT
    REPLACE(REGEXP_REPLACE(review_id,        '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS review_id,
    REPLACE(REGEXP_REPLACE(product_id,       '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS product_id,
    REPLACE(REGEXP_REPLACE(reviewer_id,      '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS reviewer_id,
    REPLACE(REGEXP_REPLACE(reviewer_name,    '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS reviewer_name,
    REPLACE(REGEXP_REPLACE(rating,           '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS rating,
    REPLACE(REGEXP_REPLACE(review_date,      '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS review_date,
    REPLACE(REGEXP_REPLACE(review_title,     '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS review_title,
    REPLACE(REGEXP_REPLACE(review_text,      '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS review_text,
    REPLACE(REGEXP_REPLACE(product_category, '<[^>]+>', ''), '<br />', ' ')::VARCHAR         AS product_category
FROM anycompany_lab.bronze.product_reviews;

select * from anycompany_lab.silver.product_reviews limit 10

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.silver.product_reviews_clean AS
SELECT
    -- ID numérique au début
    REGEXP_SUBSTR(review_id, '^[0-9]+') AS review_id,

    -- product_id type Amazon (B00...)
    REGEXP_SUBSTR(review_id, 'B[0-9A-Z]{9}') AS product_id,

    -- reviewer_id (A....)
    REGEXP_SUBSTR(review_id, 'A[0-9A-Z]{10,}') AS reviewer_id,

    -- date fiable
    REGEXP_SUBSTR(review_id, '\\d{4}-\\d{2}-\\d{2}') AS review_date,

    -- tout le reste = texte brut (pour Cortex c’est suffisant)
    review_id AS full_raw_text

FROM anycompany_lab.bronze.product_reviews;

In [ ]:
SELECT *
FROM anycompany_lab.silver.product_reviews_clean
LIMIT 20;

In [ ]:
-- logistics_and_shipping
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.logistics_and_shipping_clean
AS
SELECT
    shipment_id::VARCHAR             AS shipment_id,
    order_id::INTEGER                AS order_id,
    ship_date::DATE                  AS ship_date,
    estimated_delivery::DATE         AS estimated_delivery,
    shipping_method::VARCHAR         AS shipping_method,
    status::VARCHAR                  AS status,
    shipping_cost::NUMBER(8,2)       AS shipping_cost,
    destination_region::VARCHAR      AS destination_region,
    destination_country::VARCHAR     AS destination_country,
    carrier::VARCHAR                 AS carrier
FROM anycompany_lab.bronze.logistics_and_shipping;

select * from anycompany_lab.silver.logistics_and_shipping_clean limit 10

In [ ]:

-- supplier_information
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.supplier_information_clean
AS
SELECT
    supplier_id::VARCHAR             AS supplier_id,
    supplier_name::VARCHAR           AS supplier_name,
    product_category::VARCHAR        AS product_category,
    region::VARCHAR                  AS region,
    country::VARCHAR                 AS country,
    city::VARCHAR                    AS city,
    lead_time::INTEGER               AS lead_time,
    reliability_score::NUMBER(4,2)   AS reliability_score,
    quality_rating::VARCHAR          AS quality_rating
FROM anycompany_lab.bronze.supplier_information;

select * from anycompany_lab.silver.supplier_information_clean limit 10


In [ ]:
-- employee_records
CREATE TABLE IF NOT EXISTS anycompany_lab.silver.employee_records_clean
AS
SELECT
    employee_id::VARCHAR             AS employee_id,
    name::VARCHAR                    AS name,
    date_of_birth::DATE              AS date_of_birth,
    hire_date::DATE                  AS hire_date,
    department::VARCHAR              AS department,
    job_title::VARCHAR               AS job_title,
    salary::NUMBER(12,2)             AS salary,
    region::VARCHAR                  AS region,
    country::VARCHAR                 AS country,
    email::VARCHAR                   AS email
FROM anycompany_lab.bronze.employee_records;

select * from anycompany_lab.silver.employee_records_clean limit 10

# Phase 2

## Partie 2.1

1. `customer_demographics_clean`

* Périmètre métier : profil sociodémographique des clients (âge, genre, localisation, revenus, statut marital)
* Colonnes clés : `customer_id` (PK), `region`, `country`
* Volumes et périodes : pas de date de création ou mise à jour, impossible de déterminer la période couverte
* Anomalies : aucune colonne permettant de relier un client à ses achats ou interactions



2. `customer_service_interactions_clean`

* Périmètre métier : historique des contacts SAV (type, durée, satisfaction, statut de résolution)
* Colonnes clés : `interaction_id` (PK), `interaction_date`
* Volumes et périodes : période couverte visible via `interaction_date`
* Anomalies : pas de `customer_id`, impossible de relier une interaction à un client



3. `financial_transactions_clean`

* Périmètre métier : mouvements financiers (ventes, remboursements, investissements, paiements de taxes)
* Colonnes clés : `transaction_id` (PK), `transaction_date`, `account_code`
* Volumes et périodes : période couverte via `transaction_date`
* Anomalies : pas de `customer_id` ni `order_id`, `entity` est en texte libre et non une clé structurée



4. `promotions_clean`

* Périmètre métier : promotions par catégorie produit, région et période
* Colonnes clés : `promotion_id` (PK), `start_date`, `end_date`
* Volumes et périodes : période couverte via `start_date` et `end_date`
* Anomalies : pas de `product_id`, lien uniquement via `product_category`, trop large pour mesurer un impact précis



5. `marketing_campaigns_clean`

* Périmètre métier : campagnes marketing (type, audience cible, budget, portée, taux de conversion)
* Colonnes clés : `campaign_id` (PK), `start_date`, `end_date`
* Volumes et périodes : période couverte via `start_date` et `end_date`
* Anomalies : structure propre, vérifier la cohérence entre `conversion_rate` et `reach`



6. `product_reviews_clean`

* Périmètre métier : avis clients sur les produits (note, texte, date, catégorie)
* Colonnes clés : `review_id` (PK), `product_id`
* Volumes et périodes : `review_date` en VARCHAR, période non exploitable directement
* Anomalies : données issues de la bronze non correctement typées, `rating` et `review_date` en texte libre, toutes les colonnes en VARCHAR, pas de `customer_id`



7. `logistics_and_shipping_clean`

* Périmètre métier : suivi des expéditions (méthode, statut, coût, délais, transporteur)
* Colonnes clés : `shipment_id` (PK), `order_id`, `ship_date`
* Volumes et périodes : période couverte via `ship_date` et `estimated_delivery`
* Anomalies : structure propre, vérifier que `estimated_delivery` est postérieur à `ship_date`



8. `supplier_information_clean`

* Périmètre métier : référentiel fournisseurs (localisation, délai, fiabilité, qualité)
* Colonnes clés : `supplier_id` (PK), `product_category`
* Volumes et périodes : pas de date, référentiel statique
* Anomalies : structure propre, vérifier que `reliability_score` est compris entre 0 et 1



9. `employee_records_clean`

* Périmètre métier : données RH (poste, département, salaire, localisation, contact)
* Colonnes clés : `employee_id` (PK), `hire_date`
* Volumes et périodes : période couverte via `hire_date`
* Anomalies : structure propre, vérifier que `hire_date` est postérieur à `date_of_birth` et la validité des emails



10. `inventory_clean`

* Périmètre métier : niveaux de stock par produit, entrepôt et région
* Colonnes clés : `product_id`, `warehouse`
* Volumes et périodes : période couverte via `last_restock_date`
* Anomalies : pas de clé étrangère vers une table produits, cohérence de `product_id` à vérifier avec les autres tables



11. `store_locations_clean`

* Périmètre métier : référentiel des magasins physiques (type, taille, localisation, effectif)
* Colonnes clés : `store_id` (PK)
* Volumes et périodes : pas de date, référentiel statique
* Anomalies : `postal_code` converti en entier, perte des zéros initiaux et incompatibilité avec certains formats internationaux




## Partie 2.2 – Analyses exploratoires descriptives

### 2.2.1 - Évolution des ventes dans le temps
Tendances mensuelles, croissance du chiffre d'affaires et saisonnalité

In [ ]:
-- CA et volume de transactions par mois
SELECT
    DATE_TRUNC('month', transaction_date)    AS month,
    COUNT(transaction_id)                    AS nb_transactions,
    ROUND(SUM(amount), 2)                    AS total_revenue,
    ROUND(AVG(amount), 2)                    AS avg_basket,
    ROUND(MEDIAN(amount), 2)                 AS median_basket
FROM anycompany_lab.silver.financial_transactions_clean
WHERE amount > 0
GROUP BY 1
ORDER BY 1;

In [ ]:
-- Taux de croissance mois sur mois
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', transaction_date) AS month,
        SUM(amount)                            AS revenue
    FROM anycompany_lab.silver.financial_transactions_clean
    WHERE amount > 0
    GROUP BY 1
)
SELECT
    month,
    ROUND(revenue, 2)                        AS revenue,
    ROUND(LAG(revenue) OVER (ORDER BY month), 2) AS prev_month_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY month))
        / NULLIF(LAG(revenue) OVER (ORDER BY month), 0) * 100
    , 1)                                       AS mom_growth_pct
FROM monthly
ORDER BY 1;

In [ ]:
-- Distribution du CA par jour et par mois (matrice saisonnalité)
SELECT
    MONTHNAME(transaction_date)              AS month_name,
    MONTH(transaction_date)                  AS month_num,
    DAYOFWEEKISO(transaction_date)           AS dow_iso,
    DAYNAME(transaction_date)               AS day_name,
    COUNT(*)                                AS nb_transactions,
    ROUND(SUM(amount), 2)                  AS total_revenue
FROM anycompany_lab.silver.financial_transactions_clean
WHERE amount > 0
GROUP BY 1, 2, 3, 4
ORDER BY month_num, dow_iso;

### 2.2.2 - Performance par produit, catégorie et région
Top produits, stocks critiques, satisfaction client, performance des campagnes et magasins

In [ ]:
-- Produits sous le seuil de réassort par entrepôt
SELECT
    warehouse,
    COUNT(product_id)                        AS nb_products,
    SUM(current_stock)                       AS total_stock,
    ROUND(AVG(current_stock), 1)             AS avg_stock,
    COUNT_IF(current_stock <= reorder_point) AS products_below_reorder,
    COUNT_IF(current_stock = 0)              AS out_of_stock,
    ROUND(COUNT_IF(current_stock <= reorder_point) * 100.0
          / NULLIF(COUNT(*), 0), 1)          AS reorder_alert_pct
FROM anycompany_lab.silver.inventory_clean
GROUP BY 1
ORDER BY reorder_alert_pct DESC;


In [ ]:
-- ROI et conversion par campagne
SELECT
    campaign_id,
    DATEDIFF('day', start_date, end_date)   AS duration_days,
    budget,
    ROUND(conversion_rate * 100, 2)         AS conversion_rate_pct,
    ROUND(budget / NULLIF(conversion_rate, 0), 2) AS cost_per_conversion,
    NTILE(4) OVER (ORDER BY conversion_rate) AS performance_quartile
FROM anycompany_lab.silver.marketing_campaigns_clean
WHERE end_date >= start_date
ORDER BY conversion_rate DESC;

In [ ]:
-- KPIs magasins regroupés par pays / région
SELECT
    country,
    COUNT(store_id)                         AS nb_stores,
    SUM(employee_count)                     AS total_employees,
    ROUND(AVG(employee_count), 1)           AS avg_employees_per_store,
    SUM(square_footage)                     AS total_sqft,
    ROUND(AVG(square_footage), 0)           AS avg_sqft_per_store,
    ROUND(SUM(employee_count) * 1.0
          / NULLIF(SUM(square_footage), 0), 4) AS staff_density
FROM anycompany_lab.silver.store_locations_clean
GROUP BY 1
ORDER BY nb_stores DESC;

### 2.2.3 - Segmentation démographique des clients
Profils clients par genre, tranche d'âge, revenu et satisfaction service

In [ ]:
-- Distribution démographique (genre × tranche d'âge)
SELECT
    gender,
    CASE
        WHEN DATEDIFF('year', date_of_birth, CURRENT_DATE()) < 25 THEN '18-24'
        WHEN DATEDIFF('year', date_of_birth, CURRENT_DATE()) < 35 THEN '25-34'
        WHEN DATEDIFF('year', date_of_birth, CURRENT_DATE()) < 50 THEN '35-49'
        WHEN DATEDIFF('year', date_of_birth, CURRENT_DATE()) < 65 THEN '50-64'
        ELSE '65+'
    END                                      AS age_group,
    COUNT(customer_id)                      AS nb_customers,
    ROUND(COUNT(customer_id) * 100.0
          / SUM(COUNT(customer_id)) OVER (), 1) AS share_pct
FROM anycompany_lab.silver.customer_demographics_clean
WHERE date_of_birth IS NOT NULL
GROUP BY 1, 2
ORDER BY gender, age_group;

In [ ]:
-- Profil économique des clients
SELECT
    CASE
        WHEN annual_income < 20000  THEN '< 20k'
        WHEN annual_income < 40000  THEN '20k–40k'
        WHEN annual_income < 70000  THEN '40k–70k'
        WHEN annual_income < 100000 THEN '70k–100k'
        ELSE '100k+'
    END                                      AS income_bracket,
    COUNT(customer_id)                      AS nb_customers,
    ROUND(AVG(annual_income), 0)            AS avg_income,
    ROUND(COUNT(customer_id) * 100.0
          / SUM(COUNT(customer_id)) OVER (), 1) AS share_pct
FROM anycompany_lab.silver.customer_demographics_clean
WHERE annual_income IS NOT NULL
GROUP BY 1
ORDER BY MIN(annual_income);

In [ ]:
-- Score de satisfaction et durée selon le canal
SELECT
    interaction_type,
    COUNT(interaction_id)                   AS nb_interactions,
    ROUND(AVG(customer_satisfaction), 2)   AS avg_satisfaction,
    ROUND(AVG(duration_minutes), 1)        AS avg_duration_min,
    COUNT_IF(customer_satisfaction >= 4)   AS satisfied_count,
    COUNT_IF(customer_satisfaction <= 2)   AS unsatisfied_count,
    ROUND(COUNT_IF(customer_satisfaction >= 4) * 100.0
          / NULLIF(COUNT(*), 0), 1)         AS satisfaction_rate_pct
FROM anycompany_lab.silver.customer_service_interactions_clean
WHERE customer_satisfaction IS NOT NULL
GROUP BY 1
ORDER BY avg_satisfaction DESC;

## Partie 2.3

### 2.3.1. Ventes et Promotions
Comparaison avec/sans promotion et sensibilité des catégories

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.gold.analysis_sales_promotions AS
SELECT 
    t.transaction_id,
    t.transaction_date,
    t.region,
    t.amount,
    p.promotion_id,
    p.promotion_type,
    p.discount_percentage,
    CASE WHEN p.promotion_id IS NOT NULL THEN 'Promotion' ELSE 'Full Price' END AS sales_type
FROM anycompany_lab.silver.financial_transactions_clean t
LEFT JOIN anycompany_lab.silver.promotions_clean p 
    ON t.region = p.region 
    AND t.transaction_date BETWEEN p.start_date AND p.end_date
WHERE t.transaction_type = 'Sale';

select * from anycompany_lab.gold.analysis_sales_promotions limit 10

### 2.3.2. Marketing et Performance Commerciale
Lien campagnes ↔ ventes et efficacité

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.gold.analysis_marketing_performance AS
SELECT 
    m.campaign_name,
    m.product_category,
    m.budget,
    m.reach,
    m.conversion_rate,
    SUM(t.amount) AS revenue_generated,
    (SUM(t.amount) / NULLIF(m.budget, 0)) AS roi
FROM anycompany_lab.silver.marketing_campaigns_clean m
LEFT JOIN anycompany_lab.silver.financial_transactions_clean t
    ON m.region = t.region 
    AND m.product_category = t.entity -- Note: Vérifiez si 'entity' contient la catégorie
    AND t.transaction_date BETWEEN m.start_date AND m.end_date
GROUP BY 1, 2, 3, 4, 5;

select * from anycompany_lab.gold.analysis_marketing_performance limit 10

### 2.3.3. Expérience Client
Impact des avis et du service client

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.silver.analysis_experience_cortex AS
SELECT 
    product_id,
    COUNT(*) AS total_reviews,
    AVG(SNOWFLAKE.CORTEX.SENTIMENT(full_raw_text)) AS avg_sentiment
FROM anycompany_lab.silver.product_reviews_clean
GROUP BY product_id;

select * from anycompany_lab.silver.analysis_experience_cortex

In [ ]:
WITH reviews_enriched AS (
    SELECT
        product_id,
        full_raw_text,

        -- sentiment
        SNOWFLAKE.CORTEX.SENTIMENT(full_raw_text) AS sentiment,

        -- classification simple (à adapter selon ton dataset)
        CASE 
            WHEN LOWER(full_raw_text) LIKE '%good%' THEN 'good'
            WHEN LOWER(full_raw_text) LIKE '%bad%' THEN 'bad'
        END AS derived_category

    FROM anycompany_lab.silver.product_reviews_clean
),

reviews_agg AS (
    SELECT
        derived_category,
        COUNT(*) AS total_reviews,
        AVG(sentiment) AS avg_sentiment
    FROM reviews_enriched
    GROUP BY derived_category
),

service_agg AS (
    SELECT
        LOWER(issue_category) AS category,
        COUNT(*) AS total_tickets,
        AVG(customer_satisfaction) AS avg_service_satisfaction
    FROM anycompany_lab.silver.customer_service_interactions_clean
    GROUP BY LOWER(issue_category)
),

sales_agg AS (
    SELECT
        LOWER(entity) AS category,
        SUM(amount) AS total_sales
    FROM anycompany_lab.silver.financial_transactions_clean
    GROUP BY LOWER(entity)
)

SELECT
    r.derived_category,

    r.total_reviews,
    r.avg_sentiment,

    s.total_tickets,
    s.avg_service_satisfaction,

    f.total_sales

FROM reviews_agg r
LEFT JOIN service_agg s
    ON r.derived_category = s.category
LEFT JOIN sales_agg f
    ON r.derived_category = f.category

ORDER BY r.avg_sentiment DESC;

### 2.3.4. Opérations et Logistique
Ruptures de stock et délais de livraison

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.silver.analysis_supply_chain AS
SELECT 
    i.product_id,
    i.warehouse,
    i.current_stock,
    i.reorder_point,
    CASE 
        WHEN i.current_stock <= i.reorder_point THEN 'RISQUE DE RUPTURE' 
        ELSE 'STOCK OK' 
    END AS alert_level,
    s.supplier_name,
    s.lead_time AS supplier_lead_time,
    s.reliability_score
FROM anycompany_lab.silver.inventory_clean i
LEFT JOIN anycompany_lab.silver.supplier_information_clean s
    ON i.product_category = s.product_category 
    AND i.region = s.region;

    select * from anycompany_lab.silver.analysis_supply_chain limit 10

# Phase 3

création du schéma analytics

In [ ]:
create or replace schema anycompany_lab.analytics

## PARTIE 3.1 — DATA PRODUCT

### 1. Table centrale

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.analytics.fact_sales_enriched AS
SELECT
    t.transaction_id,
    t.transaction_date,
    t.amount,
    t.payment_method,

    -- dimensions
    t.region,
    t.entity AS product_category,

    -- promotions
    p.promotion_type,
    p.discount_percentage,

    -- marketing
    m.campaign_name,
    m.campaign_type,
    m.budget,

    -- indicateur clé
    CASE 
        WHEN p.promotion_id IS NOT NULL THEN 1 
        ELSE 0 
    END AS is_promo_active

FROM anycompany_lab.silver.financial_transactions_clean t

LEFT JOIN anycompany_lab.silver.promotions_clean p
    ON LOWER(t.entity) = LOWER(p.product_category)
    AND t.transaction_date BETWEEN p.start_date AND p.end_date

LEFT JOIN anycompany_lab.silver.marketing_campaigns_clean m
    ON LOWER(t.entity) = LOWER(m.product_category)
    AND t.region = m.region
    AND t.transaction_date BETWEEN m.start_date AND m.end_date;

### 2. Table client

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.analytics.dim_customers_enriched AS
SELECT
    customer_id,
    age,
    gender,
    region,
    country,
    annual_income,

    -- segmentation simple
    CASE 
        WHEN annual_income < 30000 THEN 'low'
        WHEN annual_income BETWEEN 30000 AND 70000 THEN 'medium'
        ELSE 'high'
    END AS income_segment

FROM (
    SELECT *,
        DATEDIFF(year, date_of_birth, CURRENT_DATE) AS age
    FROM anycompany_lab.silver.customer_demographics_clean
);

### 3. Table features (ML)

In [ ]:
CREATE OR REPLACE TABLE anycompany_lab.analytics.features_marketing AS
SELECT
    region,
    product_category,

    COUNT(*) AS total_transactions,
    SUM(amount) AS total_revenue,
    AVG(amount) AS avg_basket,

    AVG(is_promo_active) AS promo_ratio,

    COUNT(DISTINCT campaign_name) AS nb_campaigns,

    -- temporal features
    MONTH(transaction_date) AS month,
    DAYOFWEEK(transaction_date) AS day_of_week

FROM anycompany_lab.analytics.fact_sales_enriched
GROUP BY region, product_category, month, day_of_week;

## PARTIE 3.2 — FEATURE ENGINEERING

In [ ]:
-- récence (proxy simple)
DATEDIFF(day, MAX(transaction_date), CURRENT_DATE) AS recency

-- fréquence
COUNT(*) AS frequency

-- valeur client
SUM(amount) AS monetary